# 4. Normalization comparison results

Consolidated replacement for `4. results_z.ipynb`, `4. results_none.ipynb`, `4. results_rk.ipynb`, and `4. results_bin.ipynb`.

It evaluates the same validation datasets for each normalization result directory and summarizes protein-level, sample/tissue-level, and high-performance-protein behavior in one concise workflow.

In [ ]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd()
if str(NOTEBOOK_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_DIR))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

import src.utils_val as uv
from src.utils_val import (
    evaluate_norm_config,
    plot_norm_summary,
    plot_norm_protein_distributions,
    build_top_protein_table,
    high_performance_counts,
    plot_norm_tissue_heatmap,
)

sns.set_theme(style="whitegrid", context="notebook")
plt.rcParams.update({"figure.dpi": 130, "savefig.dpi": 180, "axes.titleweight": "bold"})

## 1. Configuration

The paths below preserve the original notebooks. Every normalization writes its plots inside its own results folder (`<results_dir>/plots`).

In [ ]:
VAL_DATA_DIR = Path("../val_data")
OUTPUT_DIR = Path("../results_val/plots_norms_unified")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

VALIDATION_DATASETS = ["iTRAQ", "LabelFree", "TMT"]

NORM_CONFIGS = [
    {"label": "zscore", "results_dir": Path("../results_val"), "plot_dir": Path("../results_val/plots_z")},
    {"label": "none", "results_dir": Path("../results_none"), "plot_dir": Path("../results_none/plots")},
    {"label": "rank", "results_dir": Path("../results_rk"), "plot_dir": Path("../results_rk/plots")},
    {"label": "bin", "results_dir": Path("../results_bin"), "plot_dir": Path("../results_bin/plots")},
]

for cfg in NORM_CONFIGS:
    cfg["plot_dir"].mkdir(parents=True, exist_ok=True)

# The original 4.* notebooks imputed missing selected RNA features as zero.
IMPUTE_MISSING_FEATURES = True

pd.DataFrame(NORM_CONFIGS)

## 2. Evaluate each normalization directory

In [ ]:
norm_bundles = []
for cfg in NORM_CONFIGS:
    print("\n" + "=" * 100)
    print(f"Normalization: {cfg['label']} | models: {cfg['results_dir']}")
    print("=" * 100)
    bundle = evaluate_norm_config(
        cfg["label"],
        cfg["results_dir"],
        VAL_DATA_DIR,
        datasets=VALIDATION_DATASETS,
        impute_missing_features=IMPUTE_MISSING_FEATURES,
    )
    norm_bundles.append(bundle)

summary_df = pd.concat([b["summary"] for b in norm_bundles if len(b["summary"])], ignore_index=True) if norm_bundles else pd.DataFrame()
protein_metrics_df = pd.concat([b["protein"] for b in norm_bundles if len(b["protein"])], ignore_index=True) if norm_bundles else pd.DataFrame()
tissue_metrics_df = pd.concat([b["tissue"] for b in norm_bundles if len(b["tissue"])], ignore_index=True) if norm_bundles else pd.DataFrame()

print("Summary rows:", len(summary_df))
display(summary_df.round(4))

## 3. Overview plots

In [ ]:
fig = plot_norm_summary(summary_df)
if fig is not None:
    fig.savefig(OUTPUT_DIR / "norm_summary.png", bbox_inches="tight")

In [ ]:
plot_norm_protein_distributions(protein_metrics_df)

## 4. Tissue-level comparison

In [ ]:
plot_norm_tissue_heatmap(tissue_metrics_df, metric="MSE")

## 5. High-performance proteins and top lists

In [ ]:
high_perf = high_performance_counts(protein_metrics_df, threshold=0.5)
print("Proteins above 0.5 Pearson/Cosine by normalization and dataset")
display(high_perf.round(4))

top_proteins = build_top_protein_table(protein_metrics_df, top_n=20)
print("Top 20 proteins per normalization/dataset/metric")
display(top_proteins.round(4))

## 6. Optional detailed dashboards

The original split notebooks generated multiple per-dataset dashboard PNGs. This consolidated notebook keeps the core plots above by default. Set `SAVE_DETAILED_TABLES = True` to export the summary, protein metrics, tissue metrics, high-performance table, and top-protein table.

In [ ]:
SAVE_DETAILED_TABLES = False
if SAVE_DETAILED_TABLES:
    summary_df.to_csv(OUTPUT_DIR / "norm_summary.csv", index=False)
    protein_metrics_df.to_csv(OUTPUT_DIR / "norm_protein_metrics.csv", index=False)
    tissue_metrics_df.to_csv(OUTPUT_DIR / "norm_tissue_metrics.csv", index=False)
    high_perf.to_csv(OUTPUT_DIR / "norm_high_performance_counts.csv", index=False)
    top_proteins.to_csv(OUTPUT_DIR / "norm_top_proteins.csv", index=False)
    print("Saved tables to", OUTPUT_DIR.resolve())